In [1]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.cluster import KMeans
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [24]:
DATA_PATH = Path("../data/processed/lastfm_scrobbles_clean_tags_final.parquet")

In [25]:
model_df = pd.read_parquet(DATA_PATH)

In [26]:
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(model_df["tags_genre_mood"])

In [27]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X)

In [30]:
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_reduced)
model_df["cluster"] = clusters

In [31]:
def get_top_tags_per_cluster(X, clusters, feature_names, top_n=10):
    cluster_tags = {}
    
    for cluster in np.unique(clusters):
        idx = clusters == cluster
        mean_vals = X[idx].mean(axis=0)
        top_idx = np.argsort(mean_vals)[-top_n:]
        cluster_tags[cluster] = [feature_names[i] for i in top_idx]
        
    return cluster_tags

top_tags = get_top_tags_per_cluster(X, clusters, mlb.classes_)

for k, v in top_tags.items():
    print(f"Cluster {k}: {v}")

Cluster 0: ['composers', 'score', 'modern classical', 'ambient', 'piano', 'contemporary classical', 'soundtrack', 'classical', 'composer', 'instrumental']
Cluster 1: ['pop rock', 'new wave', 'post punk revival', 'folk', 'garage rock', 'shoegaze', 'post punk', 'britpop', 'indie', 'alternative']
Cluster 2: ['pop rock', 'progressive rock', 'folk', 'psychedelic', 'folk rock', 'psychedelic rock', 'blues rock', 'blues', 'hard rock', 'classic rock']
Cluster 3: ['art pop', 'new wave', 'electropop', 'psychedelic', 'experimental', 'synthpop', 'post punk', 'lo fi', 'shoegaze', 'indie']
Cluster 4: ['hardcore', 'garage rock', 'indie', 'hardcore punk', 'pop punk', 'polish rock', 'post punk', 'alternative', 'punk rock', 'punk']
Cluster 5: ['art pop', 'drone', 'indie', 'downtempo', 'psychedelic', 'idm', 'post rock', 'instrumental', 'ambient', 'experimental']
Cluster 6: ['country', 'lo fi', 'alt country', 'americana', 'folk rock', 'alternative', 'acoustic', 'indie folk', 'indie', 'folk']
Cluster 7: ['i

In [34]:
cluster_names = {
    0: "Classical / Soundtrack",
    1: "UK Indie / Post-Punk",
    2: "Classic / Psychedelic Rock",
    3: "Art Pop / Indietronica",
    4: "Punk / Hardcore",
    5: "Experimental Ambient",
    6: "Folk / Americana",
    7: "Chill Electronic / Trip-Hop",
    8: "Mixed Groove / Alt",
    9: "Art Rock / Experimental Indie"
}

In [35]:
model_df["cluster_name"] = model_df["cluster"].map(cluster_names)
model_df["cluster_name"].value_counts()

cluster_name
Mixed Groove / Alt               15377
UK Indie / Post-Punk             14825
Art Pop / Indietronica            9409
Folk / Americana                  7896
Experimental Ambient              6832
Art Rock / Experimental Indie     4807
Classic / Psychedelic Rock        4728
Chill Electronic / Trip-Hop       3405
Punk / Hardcore                   3118
Classical / Soundtrack            2426
Name: count, dtype: int64

In [10]:
# why not audio features?
# Spotify audio features were unavailable due to API limitations (403 errors), so I used Last.fm tags as a semantic representation of music, which actually captures genre and mood more directly.